## Packages

In [1]:
from datetime import datetime, timedelta
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np

## Parameters

In [2]:
# ============================================================
# 1. PARÁMETROS DEL ANÁLISIS
# ============================================================
n=16752
new_n = 8164 # nuevo número de días
fecha_actual = "2025-12-01"
# Parámetros de FAI
window_minutes=30
fai_temp_range = (8, 35)
fai_em_jump = 0.015
fai_em_threshold_d=fai_em_jump
date_column = "date"
duration = True
FAI_duration = 4
exclude_last_minutes = 20 # minutos al final del día para excluir FAIs
filter_flare_coincidence = True
method = "filteredend_nooverlap" # "all", "true", "filtered", "filteredend" o "filteredend_nooverlap"
flares="ABCMX" # "valid", "no_A", "no_A_B", 
                                  # "no_A_B_nonoverlap" o "no_A_B_nonoverlap_wstart"
                                  # "M_X_nonoverlap_wstart"
nooverlap = True
min_gap_minutes_overlap = 20 #FILTRO EXTRA: remover FAI que ocurren tras un flare en [end, end + min_gap_minutes_overlap] 

# Formato de threshold para nombres de carpetas
threshold_str = f"{fai_em_jump}"   # "0.005" # convertir a string sin notación científica
fai_em_threshold_d = threshold_str.replace(".", "d") # reemplazar el punto por 'd'

## Folder

In [3]:
analysis_dir = os.path.join(f"Data_for_{n}_days/Analysis_FAI")
# Crear carpeta de análisis (si no existe) y mostrar mensaje
if os.path.exists(analysis_dir ):
    print(f"⚠️ La carpeta de análisis ya existía: {analysis_dir }")
else:
    os.makedirs(analysis_dir , exist_ok=True)
    print(f"📁 Carpeta de análisis creada: {analysis_dir}")

# Crear subcarpeta para guardar resultados de análisis
analysis_esp = os.path.join(analysis_dir, f"Analysis_FAI_{flares}_T{fai_temp_range[0]}-{fai_temp_range[1]}_dEM{fai_em_threshold_d}_dur{FAI_duration}min")
# Crear carpeta de análisis (si no existe) y mostrar mensaje
if os.path.exists(analysis_esp ):
    print(f"⚠️ La carpeta de análisis ya existía: {analysis_esp }")
else:
    os.makedirs(analysis_esp , exist_ok=True)
    print(f"📁 Carpeta de análisis creada: {analysis_esp}")

⚠️ La carpeta de análisis ya existía: Data_for_16752_days/Analysis_FAI
⚠️ La carpeta de análisis ya existía: Data_for_16752_days/Analysis_FAI/Analysis_FAI_ABCMX_T8-35_dEM0d015_dur4min


## Valid Data

In [4]:
input_dir = f"Data_for_{n}_days"

# Path of cleaned data in csv
csv_path_full_valid = f"{input_dir}/df_full_{new_n}_valid.csv"
# Path of valid flares: 
csv_path_all_flares = f"{input_dir}/all_df_flare_data_{n}.csv"
csv_path_flares_valid = f"{input_dir}/df_flares_{new_n}_valid.csv"
csv_path_flares = f"{input_dir}/df_flares_{new_n}_{flares}.csv"

#csv_path_flares = f"{input_dir}/all_df_flares_{n}_cleaned.csv"
# Path of df_full
csv_path_full = f"{input_dir}/all_df_full_{n}.csv"

# Verificar existencia de archivos y avisar
if not os.path.exists(csv_path_full_valid):
    print(f"⚠️ No se encontró el archivo GOES: {csv_path_full_valid}")
else:
    print(f"✅ Archivo GOES encontrado: {csv_path_full_valid}")

if not os.path.exists(csv_path_all_flares):
    print(f"⚠️ No se encontró el archivo de flares: {csv_path_all_flares}")
else:
    print(f"✅ Archivo de flares encontrado: {csv_path_all_flares}")
          
if not os.path.exists(csv_path_flares_valid):
    print(f"⚠️ No se encontró el archivo de flares: {csv_path_flares_valid}")
else:
    print(f"✅ Archivo de flares encontrado: {csv_path_flares_valid}")

if not os.path.exists(csv_path_flares):
    print(f"⚠️ No se encontró el archivo de flares: {csv_path_flares}")
else:
    print(f"✅ Archivo de flares encontrado: {csv_path_flares}")

if not os.path.exists(csv_path_full):
    print(f"⚠️ No se encontró el archivo de flares: {csv_path_full}")
else:
    print(f"✅ Archivo de flares encontrado: {csv_path_full}")

df_full_valid = pd.read_csv(csv_path_full_valid)
df_flares_all = pd.read_csv(csv_path_all_flares)
df_flares_valid = pd.read_csv(csv_path_flares)
df_flares = pd.read_csv(csv_path_flares)
df_full = pd.read_csv(csv_path_full)

✅ Archivo GOES encontrado: Data_for_16752_days/df_full_8164_valid.csv
✅ Archivo de flares encontrado: Data_for_16752_days/all_df_flare_data_16752.csv
✅ Archivo de flares encontrado: Data_for_16752_days/df_flares_8164_valid.csv
✅ Archivo de flares encontrado: Data_for_16752_days/df_flares_8164_ABCMX.csv
✅ Archivo de flares encontrado: Data_for_16752_days/all_df_full_16752.csv


## Data Method

In [5]:
# Path of anticipation_time
csv_path_anticipation_time = f"{analysis_esp}/df_anticipation_time_{method}_(W_{window_minutes})_T{fai_temp_range[0]}-{fai_temp_range[1]}_EM{fai_em_threshold_d}_dur{FAI_duration}min.csv"
# Path of fai_all
csv_path_fai_all = f"{analysis_esp}/df_fai_all_T{fai_temp_range[0]}-{fai_temp_range[1]}_EM{fai_em_threshold_d}_dur{FAI_duration}min.csv"
# Path of fai_selected
csv_path_fai_selected = f"{analysis_esp}/df_fai_{method}_T{fai_temp_range[0]}-{fai_temp_range[1]}_EM{fai_em_threshold_d}_dur{FAI_duration}min.csv"
# Path of df_fai_assoc
csv_path_fai_assoc = f"{analysis_esp}/df_fai_assoc_{method}_(W_{window_minutes})_T{fai_temp_range[0]}-{fai_temp_range[1]}_EM{fai_em_threshold_d}_dur{FAI_duration}min.csv"

if not os.path.exists(csv_path_anticipation_time):
    print(f"⚠️ No se encontró el archivo de flares: {csv_path_anticipation_time}")
else:
    print(f"✅ Archivo de flares encontrado: {csv_path_anticipation_time}")
if not os.path.exists(csv_path_fai_all):
    print(f"⚠️ No se encontró el archivo de flares: {csv_path_fai_all}")
else:
    print(f"✅ Archivo de flares encontrado: {csv_path_fai_all}")
if not os.path.exists(csv_path_fai_selected):
    print(f"⚠️ No se encontró el archivo de flares: {csv_path_fai_selected}")
else:
    print(f"✅ Archivo de flares encontrado: {csv_path_fai_selected}")
if not os.path.exists(csv_path_fai_assoc):
    print(f"⚠️ No se encontró el archivo de flares: {csv_path_fai_assoc}")
else:
    print(f"✅ Archivo de flares encontrado: {csv_path_fai_assoc}")


df_anticipation_time = pd.read_csv(csv_path_anticipation_time)
df_fai_all = pd.read_csv(csv_path_fai_all)
df_fai_selected = pd.read_csv(csv_path_fai_selected)
df_fai_assoc = pd.read_csv(csv_path_fai_assoc)



✅ Archivo de flares encontrado: Data_for_16752_days/Analysis_FAI/Analysis_FAI_ABCMX_T8-35_dEM0d015_dur4min/df_anticipation_time_filteredend_nooverlap_(W_30)_T8-35_EM0d015_dur4min.csv
✅ Archivo de flares encontrado: Data_for_16752_days/Analysis_FAI/Analysis_FAI_ABCMX_T8-35_dEM0d015_dur4min/df_fai_all_T8-35_EM0d015_dur4min.csv
✅ Archivo de flares encontrado: Data_for_16752_days/Analysis_FAI/Analysis_FAI_ABCMX_T8-35_dEM0d015_dur4min/df_fai_filteredend_nooverlap_T8-35_EM0d015_dur4min.csv
✅ Archivo de flares encontrado: Data_for_16752_days/Analysis_FAI/Analysis_FAI_ABCMX_T8-35_dEM0d015_dur4min/df_fai_assoc_filteredend_nooverlap_(W_30)_T8-35_EM0d015_dur4min.csv


## dfs columns to datatime

In [6]:
df_full_valid["date"] = pd.to_datetime(df_full_valid["date"])
df_flares_valid["StartTime"] = pd.to_datetime(df_flares_valid["StartTime"])
df_flares_valid["PeakTime"]  = pd.to_datetime(df_flares_valid["PeakTime"])
df_flares_valid["EndTime"]   = pd.to_datetime(df_flares_valid["EndTime"])
df_anticipation_time["StartTime"] = pd.to_datetime(df_anticipation_time["StartTime"])
df_anticipation_time["PeakTime"]  = pd.to_datetime(df_anticipation_time["PeakTime"])
df_anticipation_time["EndTime"]   = pd.to_datetime(df_anticipation_time["EndTime"])
df_fai_selected["date"] = pd.to_datetime(df_fai_selected["date"])

## Search

In [7]:
df_full_valid.head()

,date,observatory,xrsa,xrsb,xrsa_corr,xrsb_corr,T_cor,EM_cor,T_phot,EM_phot,EM_cor_norm,EM_phot_norm,month,day,GOES_missing
0,1980-04-23 00:12:00,GOES-2,7.942158e-09,8.763737e-07,1.587264e-09,4.015794e-08,7.060729,3.690231e+46,6.135368,1.300606e+47,0.003690,0.013006,4,1980-04-23,False
1,1980-04-23 00:13:00,GOES-2,7.927400e-09,8.802400e-07,1.688900e-09,4.180504e-08,7.130107,3.764167e+46,6.197556,1.320373e+47,0.003764,0.013204,4,1980-04-23,False
2,1980-04-23 00:14:00,GOES-2,7.772947e-09,8.835316e-07,1.474247e-09,4.361658e-08,6.588273,4.663026e+46,5.715241,1.696103e+47,0.004663,0.016961,4,1980-04-23,False
3,1980-04-23 00:15:00,GOES-2,8.490200e-09,8.980400e-07,2.389410e-09,5.425579e-08,7.412348,4.516885e+46,6.451488,1.554051e+47,0.004517,0.015541,4,1980-04-23,False
4,1980-04-23 00:16:00,GOES-2,9.656500e-09,9.167300e-07,2.472850e-09,5.775001e-08,7.318921,4.930460e+46,6.367286,1.707211e+47,0.004930,0.017072,4,1980-04-23,False


In [8]:
start_date = "2022-04-01 12:00:00"
end_date   = "2022-04-01 12:12:00"

df_between = df_full_valid[df_full_valid["date"].between(start_date, end_date)]

df_between

                           


,date,observatory,xrsa,xrsb,xrsa_corr,xrsb_corr,T_cor,EM_cor,T_phot,EM_phot,EM_cor_norm,EM_phot_norm,month,day,GOES_missing


In [9]:
df_anticipation_time.head()

,Flare_ID,StartTime,PeakTime,EndTime,StartPeak,StartEnd,PeakEnd,Class,ClassLetter,ClassNumber,...,FAIalerts_StartPeak,FAIalerts_PeakEnd,FAIalerts_startEnd,AnticipationStart,AnticipationPeak,Peak_to_lastFAI,Time_since_prev_flare_end,Time_since_prev_flare_peak,RelAnticipation_Peak,RelAnticipation_Start
0,198004240553_C2.0,1980-04-24 05:50:00,1980-04-24 05:53:00,1980-04-24 06:30:00,3.0,40.0,37.0,C2.0,C,2.0,...,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,198004240932_C1.0,1980-04-24 09:29:00,1980-04-24 09:32:00,1980-04-24 09:40:00,3.0,11.0,8.0,C1.0,C,1.0,...,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,198004241815_C4.0,1980-04-24 17:59:00,1980-04-24 18:15:00,1980-04-24 19:10:00,16.0,71.0,55.0,C4.0,C,4.0,...,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,198004251747_C2.0,1980-04-25 17:29:00,1980-04-25 17:47:00,1980-04-25 18:16:00,18.0,47.0,29.0,C2.0,C,2.0,...,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,198004251943_C2.0,1980-04-25 19:35:00,1980-04-25 19:43:00,1980-04-25 20:22:00,8.0,47.0,39.0,C2.0,C,2.0,...,0,0,0,NaN,NaN,NaN,87.0,116.0,NaN,NaN


## mostrar # de alertas antes del start de mayor a menor

In [10]:
# Configurar pandas para mostrar todas las columnas
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

df_anticipation_time.sort_values(by="FAIalerts_WStart", ascending=False)[40:50]

,Flare_ID,StartTime,PeakTime,EndTime,StartPeak,StartEnd,PeakEnd,Class,ClassLetter,ClassNumber,ClassGroup,NOAA_Class,PeakFlux,Observatory,Instrument,NumSpots,NOAA_AR,Zurich,McIntosh,MtWilson,HPC_X,HPC_Y,HGS_Long,HGS_Lat,EventType,EventType2,FAIalerts_W,FAIalerts_WStart,FAIalerts_WPeak,FAIalerts_StartPeak,FAIalerts_PeakEnd,FAIalerts_startEnd,AnticipationStart,AnticipationPeak,Peak_to_lastFAI,Time_since_prev_flare_end,Time_since_prev_flare_peak,RelAnticipation_Peak,RelAnticipation_Start
42893,202305181213_M1.9,2023-05-18 12:00:00,2023-05-18 12:13:00,2023-05-18 12:22:00,13.000000,22.0,9.000000,M1.9,M,1.9,M1-4.9,NaN,NaN,GOES,GOES,NaN,0.0,NaN,NaN,NaN,0.000000,38.698687,0.00000,0.00000,FL,9,3,3,3,0,0,0,18.0,31.000000,29.000000,54.000000,68.000000,2.384615,1.384615
3905,198705191810_C9.6,1987-05-19 18:08:00,1987-05-19 18:10:00,1987-05-19 18:37:00,2.000000,29.0,27.000000,C9.6,C,9.6,C5-9.9,NaN,NaN,GOES,GOES,NaN,4811.0,NaN,NaN,NaN,-348.124800,503.341200,-25.00000,30.00000,FL,9,3,3,3,0,0,0,3.0,5.000000,3.000000,NaN,NaN,2.500000,1.500000
42601,202304141659_C5.3,2023-04-14 16:44:00,2023-04-14 16:59:00,2023-04-14 17:08:00,15.000000,24.0,9.000000,C5.3,C,5.3,C5-9.9,NaN,NaN,GOES,GOES,NaN,0.0,NaN,NaN,NaN,0.000000,95.015712,0.00000,0.00000,FL,9,3,3,3,0,0,0,28.0,43.000000,41.000000,66.000000,70.000000,2.866667,1.866667
46884,202502022336_M3.2,2025-02-02 23:33:00,2025-02-02 23:36:00,2025-02-03 00:13:00,3.000000,40.0,37.000000,M3.2,M,3.2,M1-4.9,NaN,NaN,SDO,AIA,NaN,3981.0,NaN,NaN,NaN,-441.839406,177.733302,-27.00000,5.00000,FL,9,3,3,3,0,0,0,16.0,19.000000,17.000000,45.000000,49.000000,6.333333,5.333333
1427,198212250750_X2.2,1982-12-25 07:43:00,1982-12-25 07:50:00,1982-12-25 08:43:00,7.000000,60.0,53.000000,X2.2,X,2.2,X1-4.9,NaN,NaN,GOES,GOES,NaN,4033.0,NaN,NaN,NaN,-489.226200,-205.474800,-31.00000,-14.00000,FL,9,10,3,10,7,0,7,3.0,10.000000,1.000000,69.000000,82.000000,1.428571,0.428571
221,198011060348_X9.0,1980-11-06 03:40:00,1980-11-06 03:48:00,1980-11-06 05:28:00,8.000000,108.0,100.000000,X9.0,X,9.0,X5-+,NaN,NaN,GOES,GOES,NaN,2779.0,NaN,NaN,NaN,-910.842000,-218.241600,-74.00000,-12.00000,FL,9,6,3,6,3,0,3,8.0,16.000000,5.000000,120.000000,156.000000,2.000000,1.000000
34925,201410271009_M6.7,2014-10-27 09:59:00,2014-10-27 10:09:00,2014-10-27 10:26:00,10.000000,27.0,17.000000,M6.7,M,6.7,M5-9.9,NaN,NaN,GOES,GOES,NaN,12192.0,NaN,NaN,NaN,0.000000,-81.960000,0.00000,0.00000,FL,9,9,3,9,6,0,6,7.0,17.000000,5.000000,NaN,NaN,1.700000,0.700000
20213,200007102109_M5.7,2000-07-10 21:03:18,2000-07-10 21:09:38,2000-07-10 21:39:24,6.333333,36.1,29.766667,M5.7,M,5.7,M5-9.9,NaN,0.0,Yohkoh,HXT,NaN,9070.0,NaN,NaN,NaN,560.900000,252.300000,38.68706,18.56143,FL,9,3,3,3,0,0,0,27.3,33.633333,31.633333,56.633333,64.633333,5.310526,4.310526
19304,199911141623_M2.2,1999-11-14 16:20:00,1999-11-14 16:23:00,1999-11-14 16:26:00,3.000000,6.0,3.000000,M2.2,M,2.2,M1-4.9,NaN,NaN,GOES,GOES,NaN,0.0,NaN,NaN,NaN,0.000000,-49.814460,0.00000,0.00000,FL,9,3,3,3,0,0,0,16.0,19.000000,17.000000,41.000000,43.000000,6.333333,5.333333
15399,199408170108_M1.5,1994-08-17 01:07:00,1994-08-17 01:08:00,1994-08-17 01:53:00,1.000000,46.0,45.000000,M1.5,M,1.5,M1-4.9,NaN,NaN,GOES,GOES,NaN,7765.0,NaN,NaN,NaN,526.590000,-332.445000,35.00000,-15.00000,FL,9,2,2,2,0,0,0,2.0,3.000000,2.000000,26.000000,37.000000,3.000000,2.000000


In [11]:
import numpy as np

col = df_anticipation_time["Time_since_prev_flare_end"].dropna()

print("Mean:", np.mean(col))
print("Median:", np.median(col))
print("Mode:", col.mode().values)
print("Std:", np.std(col))

print("\nPercentiles:")
for p in [5, 10, 20, 25, 50, 75]:
    print(f"{p}th:", np.percentile(col, p))


Mean: 88.2101563686872
Median: 82.0
Mode: [35.]
Std: 42.29811166551295

Percentiles:
5th: 31.0
10th: 37.0
20th: 47.0
25th: 52.0
50th: 82.0
75th: 120.0


In [12]:
import numpy as np

col = df_anticipation_time["StartEnd"].dropna()

print("Mean:", np.mean(col))
print("Median:", np.median(col))
print("Mode:", col.mode().values)
print("Std:", np.std(col))

print("\nPercentiles:")
for p in [5, 10, 20, 25, 50, 75]:
    print(f"{p}th:", np.percentile(col, p))


Mean: 22.41596130509129
Median: 14.0
Mode: [8.]
Std: 25.449486312526076

Percentiles:
5th: 6.0
10th: 7.0
20th: 8.0
25th: 9.0
50th: 14.0
75th: 25.0


In [13]:
# Configurar pandas para mostrar todas las columnas
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

df_anticipation_time.sort_values(by="Class", ascending=False)[:5]

,Flare_ID,StartTime,PeakTime,EndTime,StartPeak,StartEnd,PeakEnd,Class,ClassLetter,ClassNumber,ClassGroup,NOAA_Class,PeakFlux,Observatory,Instrument,NumSpots,NOAA_AR,Zurich,McIntosh,MtWilson,HPC_X,HPC_Y,HGS_Long,HGS_Lat,EventType,EventType2,FAIalerts_W,FAIalerts_WStart,FAIalerts_WPeak,FAIalerts_StartPeak,FAIalerts_PeakEnd,FAIalerts_startEnd,AnticipationStart,AnticipationPeak,Peak_to_lastFAI,Time_since_prev_flare_end,Time_since_prev_flare_peak,RelAnticipation_Peak,RelAnticipation_Start
9880,199103222245_X9.4,1991-03-22 22:43:00,1991-03-22 22:45:00,1991-03-22 23:17:00,2.0,34.0,32.0,X9.4,X,9.4,X5-+,NaN,NaN,GOES,GOES,NaN,6555.0,NaN,NaN,NaN,-407.6484,-327.453600,-28.0,-26.0,FL,9,0,0,0,0,0,0,NaN,NaN,NaN,52.0,148.0,NaN,NaN
27791,200612051035_X9.0,2006-12-05 10:18:00,2006-12-05 10:35:00,2006-12-05 10:45:00,17.0,27.0,10.0,X9.0,X,9.0,X5-+,NaN,NaN,GOES,GOES,NaN,10930.0,NaN,NaN,NaN,-897.1380,-120.979200,-68.0,-7.0,FL,9,3,0,3,3,0,3,NaN,10.0,8.0,81.0,84.0,0.588235,NaN
221,198011060348_X9.0,1980-11-06 03:40:00,1980-11-06 03:48:00,1980-11-06 05:28:00,8.0,108.0,100.0,X9.0,X,9.0,X5-+,NaN,NaN,GOES,GOES,NaN,2779.0,NaN,NaN,NaN,-910.8420,-218.241600,-74.0,-12.0,FL,9,6,3,6,3,0,3,8.0,16.0,5.0,120.0,156.0,2.000000,1.0
45052,202405141651_X8.7,2024-05-14 16:46:00,2024-05-14 16:51:00,2024-05-14 17:02:00,5.0,16.0,11.0,X8.7,X,8.7,X5-+,NaN,NaN,GOES,GOES,NaN,13664.0,NaN,NaN,NaN,0.0000,44.529377,0.0,0.0,FL,9,2,0,2,2,0,2,NaN,2.0,1.0,71.0,83.0,0.400000,NaN
24378,200311021725_X8.3,2003-11-02 17:03:00,2003-11-02 17:25:00,2003-11-02 17:39:00,22.0,36.0,14.0,X8.3,X,8.3,X5-+,NaN,NaN,GOES,GOES,NaN,10486.0,NaN,NaN,NaN,779.3520,-272.694000,56.0,-14.0,FL,9,9,0,9,9,0,9,NaN,17.0,9.0,NaN,NaN,0.772727,NaN


In [14]:
df_2010 = df_anticipation_time[
    df_anticipation_time["StartTime"] >= "2010-01-01"
]

df_2010.sort_values(by="Class", ascending=False)[:5]



,Flare_ID,StartTime,PeakTime,EndTime,StartPeak,StartEnd,PeakEnd,Class,ClassLetter,ClassNumber,ClassGroup,NOAA_Class,PeakFlux,Observatory,Instrument,NumSpots,NOAA_AR,Zurich,McIntosh,MtWilson,HPC_X,HPC_Y,HGS_Long,HGS_Lat,EventType,EventType2,FAIalerts_W,FAIalerts_WStart,FAIalerts_WPeak,FAIalerts_StartPeak,FAIalerts_PeakEnd,FAIalerts_startEnd,AnticipationStart,AnticipationPeak,Peak_to_lastFAI,Time_since_prev_flare_end,Time_since_prev_flare_peak,RelAnticipation_Peak,RelAnticipation_Start
45052,202405141651_X8.7,2024-05-14 16:46:00,2024-05-14 16:51:00,2024-05-14 17:02:00,5.0,16.0,11.0,X8.7,X,8.7,X5-+,NaN,NaN,GOES,GOES,NaN,13664.0,NaN,NaN,NaN,0.000,44.529377,0.0,0.0,FL,9,2,0,2,2,0,2,NaN,2.0,1.0,71.0,83.0,0.400000,NaN
30178,201108090805_X6.9,2011-08-09 07:48:00,2011-08-09 08:05:00,2011-08-09 08:08:00,17.0,20.0,3.0,X6.9,X,6.9,X5-+,NaN,NaN,GOES,GOES,NaN,11263.0,NaN,NaN,NaN,845.868,239.674800,69.0,17.0,FL,9,10,0,10,10,0,10,NaN,10.0,1.0,38.0,42.0,0.588235,NaN
45025,202405110123_X5.8,2024-05-11 01:10:00,2024-05-11 01:23:00,2024-05-11 01:39:00,13.0,29.0,16.0,X5.8,X,5.8,X5-+,NaN,NaN,GOES,GOES,NaN,13664.0,NaN,NaN,NaN,0.000,51.353786,0.0,0.0,FL,9,5,0,5,5,0,5,NaN,7.0,3.0,NaN,NaN,0.538462,NaN
30968,201203070024_X5.4,2012-03-07 00:02:00,2012-03-07 00:24:00,2012-03-07 00:40:00,22.0,38.0,16.0,X5.4,X,5.4,X5-+,NaN,NaN,GOES,GOES,NaN,11429.0,NaN,NaN,NaN,0.000,122.512200,0.0,0.0,FL,9,7,0,7,7,0,7,NaN,16.0,3.0,NaN,NaN,0.727273,NaN
48293,202511111004_X5.1,2025-11-11 09:49:00,2025-11-11 10:04:00,2025-11-11 10:17:00,15.0,28.0,13.0,X5.1,X,5.1,X5-+,NaN,NaN,GOES,GOES,NaN,14274.0,NaN,NaN,NaN,0.000,-55.500932,0.0,0.0,FL,9,4,0,4,4,0,4,NaN,4.0,1.0,81.0,86.0,0.266667,NaN


In [15]:
df_flares_all.head()

,StartTime,PeakTime,EndTime,Class,PeakFlux,Observatory,Instrument,NumSpots,NOAA_AR,NOAA_Class,Zurich,McIntosh,MtWilson,HPC_X,HPC_Y,HGS_Long,HGS_Lat,EventType,EventType2
0,1980-01-05 15:10:00,1980-01-05 15:32:00,1980-01-05 15:47:00,C9.0,NaN,GOES,GOES,NaN,2220.0,NaN,NaN,NaN,NaN,-66.5904,278.1582,-4.0,13.0,FL,9
1,1980-01-05 17:46:00,1980-01-05 18:18:00,1980-01-05 18:29:00,C3.0,NaN,GOES,GOES,NaN,2232.0,NaN,NaN,NaN,NaN,-884.0100,-111.6120,-66.0,-8.0,FL,9
2,1980-01-05 18:56:00,1980-01-05 19:17:00,1980-01-05 19:29:00,C4.0,NaN,GOES,GOES,NaN,2220.0,NaN,NaN,NaN,NaN,-49.7505,294.8538,-3.0,14.0,FL,9
3,1980-01-06 06:20:00,1980-01-06 06:22:00,1980-01-06 06:36:00,C8.0,NaN,GOES,GOES,NaN,2229.0,NaN,NaN,NaN,NaN,-527.5248,381.0576,-35.0,20.0,FL,9
4,1980-01-06 12:37:00,1980-01-06 12:37:00,1980-01-06 12:47:00,C2.0,NaN,GOES,GOES,NaN,2232.0,NaN,NaN,NaN,NaN,-779.1960,-133.9152,-54.0,-10.0,FL,9


In [16]:
df_flares_valid.head()

,Flare_ID,StartTime,PeakTime,EndTime,StartPeak,StartEnd,PeakEnd,Class,ClassLetter,ClassNumber,ClassGroup,NOAA_Class,PeakFlux,Observatory,Instrument,NumSpots,NOAA_AR,Zurich,McIntosh,MtWilson,HPC_X,HPC_Y,HGS_Long,HGS_Lat,EventType,EventType2
0,198004240553_C2.0,1980-04-24 05:50:00,1980-04-24 05:53:00,1980-04-24 06:30:00,3.0,40.0,37.0,C2.0,C,2.0,C1-4.9,NaN,NaN,GOES,GOES,NaN,2396.0,NaN,NaN,NaN,-65.09340,-137.1822,-4.0,-13.0,FL,9
1,198004240932_C1.0,1980-04-24 09:29:00,1980-04-24 09:32:00,1980-04-24 09:40:00,3.0,11.0,8.0,C1.0,C,1.0,C1-4.9,NaN,NaN,GOES,GOES,NaN,2396.0,NaN,NaN,NaN,-32.28240,-170.2584,-2.0,-15.0,FL,9
2,198004241815_C4.0,1980-04-24 17:59:00,1980-04-24 18:15:00,1980-04-24 19:10:00,16.0,71.0,55.0,C4.0,C,4.0,C1-4.9,NaN,NaN,GOES,GOES,NaN,2396.0,NaN,NaN,NaN,48.62634,-154.3746,3.0,-14.0,FL,9
3,198004251747_C2.0,1980-04-25 17:29:00,1980-04-25 17:47:00,1980-04-25 18:16:00,18.0,47.0,29.0,C2.0,C,2.0,C1-4.9,NaN,NaN,GOES,GOES,NaN,2411.0,NaN,NaN,NaN,-885.28200,-260.1894,-76.0,-17.0,FL,9
4,198004251943_C2.0,1980-04-25 19:35:00,1980-04-25 19:43:00,1980-04-25 20:22:00,8.0,47.0,39.0,C2.0,C,2.0,C1-4.9,NaN,NaN,GOES,GOES,NaN,2407.0,NaN,NaN,NaN,212.26020,-427.2402,15.0,-31.0,FL,9


In [17]:
df_flares.head()

,Flare_ID,StartTime,PeakTime,EndTime,StartPeak,StartEnd,PeakEnd,Class,ClassLetter,ClassNumber,ClassGroup,NOAA_Class,PeakFlux,Observatory,Instrument,NumSpots,NOAA_AR,Zurich,McIntosh,MtWilson,HPC_X,HPC_Y,HGS_Long,HGS_Lat,EventType,EventType2
0,198004240553_C2.0,1980-04-24 05:50:00,1980-04-24 05:53:00,1980-04-24 06:30:00,3.0,40.0,37.0,C2.0,C,2.0,C1-4.9,NaN,NaN,GOES,GOES,NaN,2396.0,NaN,NaN,NaN,-65.09340,-137.1822,-4.0,-13.0,FL,9
1,198004240932_C1.0,1980-04-24 09:29:00,1980-04-24 09:32:00,1980-04-24 09:40:00,3.0,11.0,8.0,C1.0,C,1.0,C1-4.9,NaN,NaN,GOES,GOES,NaN,2396.0,NaN,NaN,NaN,-32.28240,-170.2584,-2.0,-15.0,FL,9
2,198004241815_C4.0,1980-04-24 17:59:00,1980-04-24 18:15:00,1980-04-24 19:10:00,16.0,71.0,55.0,C4.0,C,4.0,C1-4.9,NaN,NaN,GOES,GOES,NaN,2396.0,NaN,NaN,NaN,48.62634,-154.3746,3.0,-14.0,FL,9
3,198004251747_C2.0,1980-04-25 17:29:00,1980-04-25 17:47:00,1980-04-25 18:16:00,18.0,47.0,29.0,C2.0,C,2.0,C1-4.9,NaN,NaN,GOES,GOES,NaN,2411.0,NaN,NaN,NaN,-885.28200,-260.1894,-76.0,-17.0,FL,9
4,198004251943_C2.0,1980-04-25 19:35:00,1980-04-25 19:43:00,1980-04-25 20:22:00,8.0,47.0,39.0,C2.0,C,2.0,C1-4.9,NaN,NaN,GOES,GOES,NaN,2407.0,NaN,NaN,NaN,212.26020,-427.2402,15.0,-31.0,FL,9


## contar flares filtrados

In [18]:
# Datos ABCMX
total_initial = 61514
after_class_filter = 61514
after_no_overlap = 48866
after_early_start = 48344

removed_overlap = after_class_filter - after_no_overlap
removed_early_start = after_no_overlap - after_early_start

pct_overlap = removed_overlap / total_initial * 100
pct_early_start = removed_early_start / total_initial * 100
pct_final = after_early_start / total_initial * 100

# Tabla LaTeX en inglés
latex_table_en = f"""
\\begin{{table}}[h!]
\\centering
\\caption{{Summary of flare filtering for ABCMX with percentages relative to the initial total.}}
\\begin{{tabular}}{{lcc}}
\\hline
Filter Criterion & Number of flares & Percentage (\\% of initial total) \\\\
\\hline
Initial total & {total_initial} & 100 \\\\
Removed due to overlap & {removed_overlap} & {pct_overlap:.2f} \\\\
Removed due to early-start & {removed_early_start} & {pct_early_start:.2f} \\\\
Final filtered total & {after_early_start} & {pct_final:.2f} \\\\
\\hline
\\end{{tabular}}
\\label{{tab:flare_filters_abcmx}}
\\end{{table}}
"""

print(latex_table_en)




\begin{table}[h!]
\centering
\caption{Summary of flare filtering for ABCMX with percentages relative to the initial total.}
\begin{tabular}{lcc}
\hline
Filter Criterion & Number of flares & Percentage (\% of initial total) \\
\hline
Initial total & 61514 & 100 \\
Removed due to overlap & 12648 & 20.56 \\
Removed due to early-start & 522 & 0.85 \\
Final filtered total & 48344 & 78.59 \\
\hline
\end{tabular}
\label{tab:flare_filters_abcmx}
\end{table}



## Filtro de FAIs

In [19]:
# Resumen de FAI para este set de parámetros
total_points = 1603921
fai_alert = 89310
fai_true = 16964
fai_filtered = 14207
fai_filteredend = 13980
fai_filteredend_nooverlap = 12431

# Lista de pasos y valores
steps = [
    (r"Total points in $\texttt{df\_full}$", total_points),
    (r"FAI\_alert points", fai_alert),
    (r"FAI\_true points", fai_true),
    (r"Filtered (flare coincidence)", fai_filtered),
    (r"Filtered (end-of-day)", fai_filteredend),
    (r"Filtered (post-flare no-overlap)", fai_filteredend_nooverlap)
]

# Generar tabla en LaTeX
param_set_label = r"T8-35_EM0d015_dur4min"   # raw string
latex_table = r"\begin{table}[h!]" + "\n"
latex_table += r"\centering" + "\n"
latex_table += r"\caption{{Summary of FAI filtering for the parameter set $T \in [8,35]$ MK, $\Delta\mathrm{EM} > 0.015$, Duration = 4 min, with percentages relative to total $\texttt{{df\_full}}$ points.}}" + "\n"
latex_table += r"\begin{tabular}{lcc}" + "\n"
latex_table += r"\hline" + "\n"
latex_table += r"FAI Filter Step & Number of points & Percentage (\% of total $\texttt{df\_full}$) \\" + "\n"
latex_table += r"\hline" + "\n"

for label, value in steps:
    percent = value / total_points * 100
    latex_table += f"{label} & {value:,} & {percent:.2f}\\% \\\\\n"

latex_table += r"\hline" + "\n"
latex_table += rf"\end{{tabular}}" + "\n"
latex_table += rf"\label{{tab:fai_filter_{param_set_label}}}" + "\n"
latex_table += r"\end{table}" + "\n"

# Guardar en un archivo .tex
output_tex = f"FAI_table_{param_set_label}.tex"
with open(output_tex, "w") as f:
    f.write(latex_table)

print(f"LaTeX table saved to {output_tex}")



LaTeX table saved to FAI_table_T8-35_EM0d015_dur4min.tex


## revisiòn de todos los flares descargados:

In [20]:
csv_path_flares_all = f"{input_dir}/all_df_flare_data_{n}.csv"
csv_path_flares_all_clean = f"{input_dir}/all_df_flares_{n}_cleaned.csv"



# Verificar existencia de archivos y avisar
if not os.path.exists(csv_path_flares_all):
    print(f"⚠️ No se encontró el archivo GOES: {csv_path_flares_all}")
else:
    print(f"✅ Archivo GOES encontrado: {csv_path_flares_all}")

if not os.path.exists(csv_path_flares_all_clean):
    print(f"⚠️ No se encontró el archivo de flares: {csv_path_flares_all_clean}")
else:
    print(f"✅ Archivo de flares encontrado: {csv_path_flares_all_clean}")
          


df_flares_all = pd.read_csv(csv_path_flares_all)
df_flares_all_clean = pd.read_csv(csv_path_flares_all_clean)

✅ Archivo GOES encontrado: Data_for_16752_days/all_df_flare_data_16752.csv
✅ Archivo de flares encontrado: Data_for_16752_days/all_df_flares_16752_cleaned.csv


In [21]:
df_flares_all.head()

,StartTime,PeakTime,EndTime,Class,PeakFlux,Observatory,Instrument,NumSpots,NOAA_AR,NOAA_Class,Zurich,McIntosh,MtWilson,HPC_X,HPC_Y,HGS_Long,HGS_Lat,EventType,EventType2
0,1980-01-05 15:10:00,1980-01-05 15:32:00,1980-01-05 15:47:00,C9.0,NaN,GOES,GOES,NaN,2220.0,NaN,NaN,NaN,NaN,-66.5904,278.1582,-4.0,13.0,FL,9
1,1980-01-05 17:46:00,1980-01-05 18:18:00,1980-01-05 18:29:00,C3.0,NaN,GOES,GOES,NaN,2232.0,NaN,NaN,NaN,NaN,-884.0100,-111.6120,-66.0,-8.0,FL,9
2,1980-01-05 18:56:00,1980-01-05 19:17:00,1980-01-05 19:29:00,C4.0,NaN,GOES,GOES,NaN,2220.0,NaN,NaN,NaN,NaN,-49.7505,294.8538,-3.0,14.0,FL,9
3,1980-01-06 06:20:00,1980-01-06 06:22:00,1980-01-06 06:36:00,C8.0,NaN,GOES,GOES,NaN,2229.0,NaN,NaN,NaN,NaN,-527.5248,381.0576,-35.0,20.0,FL,9
4,1980-01-06 12:37:00,1980-01-06 12:37:00,1980-01-06 12:47:00,C2.0,NaN,GOES,GOES,NaN,2232.0,NaN,NaN,NaN,NaN,-779.1960,-133.9152,-54.0,-10.0,FL,9


In [22]:
# Extraer solo la letra inicial de la clase
df_flares_all['ClassLetter'] = df_flares_all['Class'].str[0]

# Contar cuántos hay de cada letra
counts = df_flares_all['ClassLetter'].value_counts()

# Calcular porcentajes
percentages = 100 * counts / counts.sum()

# Mostrar resultados
print(percentages)


ClassLetter
C    60.995098
B    28.610962
M     8.300041
A     1.496992
X     0.595963
S     0.000944
Name: count, dtype: float64


In [23]:
df_flares_all_clean.head()

,StartTime,PeakTime,EndTime,Class,PeakFlux,Observatory,Instrument,NumSpots,NOAA_AR,NOAA_Class,Zurich,McIntosh,MtWilson,HPC_X,HPC_Y,HGS_Long,HGS_Lat,EventType,EventType2
0,1980-01-05 15:10:00,1980-01-05 15:32:00,1980-01-05 15:47:00,C9.0,NaN,GOES,GOES,NaN,2220.0,NaN,NaN,NaN,NaN,-66.5904,278.1582,-4.0,13.0,FL,9
1,1980-01-05 17:46:00,1980-01-05 18:18:00,1980-01-05 18:29:00,C3.0,NaN,GOES,GOES,NaN,2232.0,NaN,NaN,NaN,NaN,-884.0100,-111.6120,-66.0,-8.0,FL,9
2,1980-01-05 18:56:00,1980-01-05 19:17:00,1980-01-05 19:29:00,C4.0,NaN,GOES,GOES,NaN,2220.0,NaN,NaN,NaN,NaN,-49.7505,294.8538,-3.0,14.0,FL,9
3,1980-01-06 06:20:00,1980-01-06 06:22:00,1980-01-06 06:36:00,C8.0,NaN,GOES,GOES,NaN,2229.0,NaN,NaN,NaN,NaN,-527.5248,381.0576,-35.0,20.0,FL,9
4,1980-01-06 12:37:00,1980-01-06 12:37:00,1980-01-06 12:47:00,C2.0,NaN,GOES,GOES,NaN,2232.0,NaN,NaN,NaN,NaN,-779.1960,-133.9152,-54.0,-10.0,FL,9
